In [3]:
!pip install -U openai-whisper
!pip install librosa

In [6]:
import whisper
import librosa
import numpy as np
import re

# ============================================================
# Hyperparams
# ============================================================
TEMPO_MU = 125          # 130 -> 125 (너희 샘플이 전반적으로 느린 편이라 중심값 살짝 낮춤)
TEMPO_SIGMA = 40        # 30 -> 40 (느린 낭독도 너무 박살나지 않게)
ALPHA = 0.03

PITCH_LOW, PITCH_HIGH = 30, 65
BETA = 0.04

E_LOW, E_HIGH = 0.02, 0.05
GAMMA_ENERGY = 30.0

K_FILLER = 5

# emphasis detection
PITCH_Z_TH_STRONG = 1.5
ENERGY_Z_TH_STRONG = 1.0
PITCH_Z_TH_WEAK = 1.2
ENERGY_Z_TH_WEAK = 0.8
MIN_EMPH_SEC = 0.20

# emphasis score band
ER_LOW = 2.0
ER_HIGH = 8.0
DELTA_EMPH = 0.08

# pause balance
PAUSE_MU = 0.78
PAUSE_SIGMA = 0.10
TOP_DB = 30

# Feature extraction alignment
SR = 16000
HOP = 512
FRAME_LENGTH = 2048

# ============================================================
# Total / Scaling controls
# ============================================================
# gating
LAM_FLUENCY = 0.55   # 0.60 -> 0.55 (너무 가혹하지 않게)
LAM_TEMPO   = 0.20   # 0.30 -> 0.20

# floors (0점 방지)
FLOOR_PAUSE = 30.0
FLOOR_TEMPO = 20.0

# calibration: raw -> final(0~100)
# p<1이면 상위 점수 끌어올림. (test3을 80대 목표로)
CALIB_P = 0.55
CALIB_MIN = 0.0
CALIB_MAX = 75.0   # raw가 보통 0~75 근처에 몰리면 여기 맞춰 스케일

# min-mix (끄는 걸 추천)
USE_MIN_MIX = False
MIN_MIX_W = 0.40

# ============================================================
# Whisper
# ============================================================
model = whisper.load_model("small")
audio_files = ["/content/test1.wav", "/content/test2.wav", "/content/test3.wav"]

# ============================================================
# Utils
# ============================================================
def clip01_to_100(x):
    return float(np.clip(x, 0.0, 100.0))

def compute_wpm(text, speech_duration_sec):
    words = len(text.split())
    if speech_duration_sec <= 1e-9:
        return 0.0
    return words / (speech_duration_sec / 60.0)

def compute_pitch_energy(y, sr):
    f0, _, _ = librosa.pyin(y, fmin=70, fmax=400, sr=sr, hop_length=HOP)
    pitch_mean = float(np.nanmean(f0)) if np.any(~np.isnan(f0)) else float("nan")
    pitch_std  = float(np.nanstd(f0))  if np.any(~np.isnan(f0)) else float("nan")

    rms = librosa.feature.rms(y=y, frame_length=FRAME_LENGTH, hop_length=HOP)[0]
    energy_mean = float(np.mean(rms))
    energy_std  = float(np.std(rms))
    return pitch_mean, pitch_std, energy_mean, energy_std, f0, rms

def estimate_speech_time_rms(y, sr, top_db=TOP_DB):
    intervals = librosa.effects.split(
        y, top_db=top_db, frame_length=FRAME_LENGTH, hop_length=HOP
    )
    speech_samples = sum((end - start) for start, end in intervals)
    return speech_samples / sr

def count_fillers_korean(text, fillers):
    tokens = text.lower().split()
    counts = {f: 0 for f in fillers}
    for t in tokens:
        t_clean = re.sub(r"[^\w가-힣]", "", t)
        if t_clean in counts:
            counts[t_clean] += 1
    return counts

# ============================================================
# Scores
# ============================================================
def score_rate_wpm(wpm, mu=TEMPO_MU, sigma=TEMPO_SIGMA):
    s = 100.0 * (1.0 - ((wpm - mu) ** 2) / (sigma ** 2))
    return clip01_to_100(s)

def segment_wpm_variance(whisper_result):
    seg_wpms = []
    for seg in whisper_result.get("segments", []):
        start = float(seg.get("start", 0.0))
        end   = float(seg.get("end", 0.0))
        dur = max(1e-6, end - start)
        words = len(str(seg.get("text", "")).strip().split())
        if words == 0:
            continue
        seg_wpms.append(words / (dur / 60.0))
    if len(seg_wpms) < 2:
        return 0.0
    return float(np.std(seg_wpms))

def score_stability(var_wpm, alpha=ALPHA):
    return clip01_to_100(100.0 * np.exp(-alpha * var_wpm))

def score_tempo(s_rate, s_stability):
    return clip01_to_100(0.6 * s_rate + 0.4 * s_stability)

def score_pitch(pitch_std, low=PITCH_LOW, high=PITCH_HIGH, beta=BETA):
    if np.isnan(pitch_std):
        return 0.0
    if pitch_std < low:
        return clip01_to_100(100.0 * (pitch_std / low))
    if low <= pitch_std <= high:
        return 100.0
    return clip01_to_100(100.0 * np.exp(-beta * (pitch_std - high)))

def score_energy(energy_std, e_low=E_LOW, e_high=E_HIGH, gamma=GAMMA_ENERGY):
    if energy_std < e_low:
        return clip01_to_100(100.0 * (energy_std / e_low))
    if e_low <= energy_std <= e_high:
        return 100.0
    return clip01_to_100(100.0 * np.exp(-gamma * (energy_std - e_high)))

def score_fluency_filler(text, filler_counts, k=K_FILLER):
    n_words = max(1, len(text.split()))
    n_filler = sum(filler_counts.values())
    fr = n_filler / n_words
    s = 100.0 * (1.0 - k * fr)
    return clip01_to_100(s), float(fr)

# emphasis detection
def _zscore(x):
    mu = np.mean(x)
    sd = np.std(x) + 1e-8
    return (x - mu) / sd

def _count_runs(mask, min_frames):
    n = 0
    run = 0
    for v in mask:
        if v:
            run += 1
        else:
            if run >= min_frames:
                n += 1
            run = 0
    if run >= min_frames:
        n += 1
    return n

def count_emphasis_weighted(f0, rms, sr):
    frame_sec = HOP / sr
    L = min(len(f0), len(rms))
    f0 = f0[:L]
    rms = rms[:L]

    voiced = ~np.isnan(f0)
    if np.sum(voiced) < 8:
        return 0.0, 0.0, 0, 0

    f0_z = np.full(L, -np.inf, dtype=float)
    f0_z[voiced] = _zscore(f0[voiced])
    rms_z = _zscore(rms)

    strong_mask = (f0_z > PITCH_Z_TH_STRONG) & (rms_z > ENERGY_Z_TH_STRONG)
    weak_mask   = (f0_z > PITCH_Z_TH_WEAK) | (rms_z > ENERGY_Z_TH_WEAK)

    min_frames = int(np.ceil(MIN_EMPH_SEC / frame_sec))
    strong_cnt = _count_runs(strong_mask, min_frames)
    weak_cnt   = _count_runs(weak_mask, min_frames)

    extra_weak = max(0, weak_cnt - strong_cnt)
    weighted = 1.0 * strong_cnt + 0.5 * extra_weak

    duration_sec = L * frame_sec
    er_per_min = weighted / max(1e-6, duration_sec / 60.0)
    return float(weighted), float(er_per_min), int(strong_cnt), int(extra_weak)

def score_emphasis_band(er_per_min, er_low=ER_LOW, er_high=ER_HIGH, delta=DELTA_EMPH):
    if er_per_min <= 0:
        return 0.0
    if er_per_min < er_low:
        return clip01_to_100(100.0 * (er_per_min / er_low))
    if er_low <= er_per_min <= er_high:
        return 100.0
    return clip01_to_100(100.0 * np.exp(-delta * (er_per_min - er_high)))

# pause score with FLOOR (fix)
def score_pause_balance_floor(speech_time_sec, total_time_sec, mu=PAUSE_MU, sigma=PAUSE_SIGMA, floor=FLOOR_PAUSE):
    if total_time_sec <= 1e-9:
        return floor, 0.0
    pr = speech_time_sec / total_time_sec
    raw = 100.0 * (1.0 - ((pr - mu) ** 2) / (sigma ** 2))
    s = max(floor, clip01_to_100(raw))
    return float(s), float(pr)

# ============================================================
# Total + Calibration
# ============================================================
def compute_total_raw(scores):
    # floors for core to avoid min-core = 0 type collapse
    tempo = max(FLOOR_TEMPO, scores["tempo"])
    pause = max(FLOOR_PAUSE, scores["pause"])

    base_components = [tempo, scores["pitch"], scores["energy"], pause]
    base = float(np.mean(base_components))

    mult_f = (1.0 - LAM_FLUENCY * (1.0 - scores["fluency"]/100.0))
    mult_t = (1.0 - LAM_TEMPO   * (1.0 - tempo/100.0))
    mult_f = np.clip(mult_f, 0.0, 1.0)
    mult_t = np.clip(mult_t, 0.0, 1.0)

    gated = base * mult_f * mult_t

    if USE_MIN_MIX:
        min_core = float(np.min(base_components + [scores["fluency"]]))
        total_raw = (1.0 - MIN_MIX_W) * gated + MIN_MIX_W * min_core
        return float(total_raw), float(base), float(gated), float(min_core), float(mult_f), float(mult_t)

    return float(gated), float(base), float(gated), float(np.min(base_components)), float(mult_f), float(mult_t)

def calibrate_to_100(raw, p=CALIB_P, lo=CALIB_MIN, hi=CALIB_MAX):
    """
    raw를 [lo, hi]로 정규화한 뒤 concave power로 상위권 끌어올림.
    - raw=hi면 100점
    - raw가 중간이어도 p<1이면 점수는 더 높게 나옴
    """
    x = (raw - lo) / max(1e-9, (hi - lo))
    x = np.clip(x, 0.0, 1.0)
    return float(100.0 * (x ** p))

# ============================================================
# Run
# ============================================================
for audio_path in audio_files:
    print(f"\n=== {audio_path} ===")

    # audio load
    y, sr = librosa.load(audio_path, sr=SR)
    total_dur = float(librosa.get_duration(y=y, sr=sr))

    # ASR
    result = model.transcribe(audio_path, language="ko")
    text = result["text"].strip()

    # WPM
    wpm = compute_wpm(text, total_dur)

    # pitch/energy
    pitch_mean, pitch_std, energy_mean, energy_std, f0, rms = compute_pitch_energy(y, sr)

    # fillers
    fillers = ["어", "음", "그"]
    filler_counts = count_fillers_korean(text, fillers)

    # tempo
    s_rate = score_rate_wpm(wpm)
    var_wpm = segment_wpm_variance(result)
    s_stability = score_stability(var_wpm)
    s_tempo = score_tempo(s_rate, s_stability)

    # pitch/energy
    s_pitch = score_pitch(pitch_std)
    s_energy = score_energy(energy_std)

    # fluency
    s_fluency, fr = score_fluency_filler(text, filler_counts)

    # emphasis
    emph_weighted, er, strong_cnt, weak_extra = count_emphasis_weighted(f0, rms, sr)
    s_emph = score_emphasis_band(er)

    # pause (RMS-based + floor)
    speech_time = estimate_speech_time_rms(y, sr, top_db=TOP_DB)
    s_pause, pr = score_pause_balance_floor(speech_time, total_dur)

    scores = {
        "tempo": s_tempo,
        "pitch": s_pitch,
        "energy": s_energy,
        "fluency": s_fluency,
        "emphasis": s_emph,
        "pause": s_pause,
    }

    total_raw, base, gated, min_core, mult_f, mult_t = compute_total_raw(scores)
    total_final = calibrate_to_100(total_raw)

    # =========================
    # Print
    # =========================
    print(f"Text: {text}")
    print(f"WPM: {wpm:.2f} | Var_WPM(segment): {var_wpm:.2f}")
    print(f"Pitch Mean: {pitch_mean:.2f} Hz | Pitch Std: {pitch_std:.2f} Hz")
    print(f"Energy Mean: {energy_mean:.4f} | Energy Std: {energy_std:.4f}")
    print(f"Filler Counts: {filler_counts} | FR: {fr:.4f}")
    print(f"Emphasis: weighted={emph_weighted:.2f} | ER(per min): {er:.2f} | strong={strong_cnt} | weak(extra)={weak_extra}")
    print(f"Pause: speech_time={speech_time:.2f}s | total_time={total_dur:.2f}s | PR={pr:.3f}")
    print("---- Scores (0~100) ----")
    print(f"Tempo Score: {s_tempo:.0f} (rate={s_rate:.0f}, stability={s_stability:.0f})")
    print(f"Pitch Score: {s_pitch:.0f}")
    print(f"Energy Score: {s_energy:.0f}")
    print(f"Fluency Score: {s_fluency:.0f}")
    print(f"Emphasis Score: {s_emph:.0f} (band {ER_LOW}-{ER_HIGH}/min)")
    print(f"Pause Balance Score: {s_pause:.0f} (floor={FLOOR_PAUSE})")
    print("---- Total ----")
    print(f"Raw total: {total_raw:.1f} | Base: {base:.1f} | mult_f={mult_f:.2f}, mult_t={mult_t:.2f}")
    print(f"Total (calibrated to 0~100): {total_final:.1f}")


=== /content/test1.wav ===
Text: 인생에 있어서 많은 것이 필요하다고는 생각지 않습니다. 튜브에 대사 인생이 뭐 별건가 달콤한 기억 하나면 그만이지처럼 고된 하루라도 소중하고 달콤한 기억 하나만 간직한다면 웃으며 살아가는 게 또 인생이라고 생각합니다. 그러나 그 말은 그 달콤한 기억, 짜릿한 성술을 위해 어떤 고통이라도 감내할 준비가 되어 있다는 말이기도 합니다. 지나간 과거에 연연하거나 다가오지 않은 미래를 두려워하지 않는 제가 되고 싶습니다. 소중한 하루를 밝게 웃으면서 살아가는 제가 되랍니다.
WPM: 77.69 | Var_WPM(segment): 6.30
Pitch Mean: 200.05 Hz | Pitch Std: 45.16 Hz
Energy Mean: 0.0426 | Energy Std: 0.0259
Filler Counts: {'어': 0, '음': 0, '그': 2} | FR: 0.0312
Emphasis: weighted=11.00 | ER(per min): 13.35 | strong=0 | weak(extra)=22
Pause: speech_time=44.93s | total_time=49.43s | PR=0.909
---- Scores (0~100) ----
Tempo Score: 33 (rate=0, stability=83)
Pitch Score: 100
Energy Score: 100
Fluency Score: 84
Emphasis Score: 65 (band 2.0-8.0/min)
Pause Balance Score: 30 (floor=30.0)
---- Total ----
Raw total: 52.1 | Base: 65.8 | mult_f=0.91, mult_t=0.87
Total (calibrated to 0~100): 81.8

=== /content/test2.wav ===
Text: 음... 사내자식 길... 어... 나설 때... 어... 갈모 하나, 거짓말 하나는... 음... 가지고 나서야 한다